In [ ]:
"""
PIPELINE STAGE 3: RadGraph Structured Extraction
------------------------------------------------
Role in Thesis:
    Shifts the evaluation from surface-level lexical similarity (BLEU/ROUGE) 
    to clinically meaningful observation-level comparisons.
    
Process:
    Converts aligned report pairs (hypothesis vs. reference) into structured 
    clinical entities and relations using the RadGraph methodology by Jain et al.
    
Outputs:
    - radgraph_extractions_{subset}_subset/ 
    Provides the basis for identifying shared, generated-only, and reference-only entities.
"""

In [ ]:
# =====================================================================
# RADGRAPH ENTITY & RELATION EXTRACTION
# =====================================================================
# Shifts the evaluation from text-based lexical overlap (e.g., BLEU, ROUGE) 
# to a clinically meaningful, observation-level representation.
# By extracting anatomical locations and clinical observations, we model 
# whether the clinically relevant content overlaps, avoiding false penalties 
# for paraphrastic variations.

# =====================================================================
# HYPOTHESIS VS. REFERENCE MAPPING
# =====================================================================
# Formats the aligned pairs into the expected RadGraph input schema:
# generated_report -> 'hypothesis'
# reference_report -> 'reference'
# This allows us to map generated clinical entities directly against the ground truth.

# =========================================
# 0. Reproducible setup cell
# =========================================
import sys
import subprocess
import importlib

REQUIRED = [
    "pandas==2.2.3",
    "tqdm==4.67.1",
    "torch>=2.1.0",
    "transformers>=4.39.0",
    "appdirs",
    "jsonpickle",
    "filelock",
    "h5py",
    "nltk",
    "dotmap",
    "pytest",
    "git+https://github.com/Stanford-AIMI/radgraph.git",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", *REQUIRED])

print("Python:", sys.executable)
print("Setup finished.")


  Cloning https://github.com/Stanford-AIMI/radgraph.git to /tmp/pip-req-build-e699c97o


  Running command git clone --filter=blob:none --quiet https://github.com/Stanford-AIMI/radgraph.git /tmp/pip-req-build-e699c97o


  Resolved https://github.com/Stanford-AIMI/radgraph.git to commit adbf1c013f88aa18c35d0c420696071013c46101
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Python: /usr/bin/python
Setup finished.


In [2]:
# =========================================
# 1. Verify imports
# =========================================
import pandas as pd
from tqdm.auto import tqdm
from radgraph import RadGraph, get_radgraph_processed_annotations

print("All imports successful.")


All imports successful.


In [3]:
# ============================================================
# 03_run_radgraph_extraction.ipynb
# Run RadGraph extraction on aligned reports
# based on outputs from 02_align_subset_to_inference.ipynb
# ============================================================

# =========================
# 1. Imports
# =========================
import os
import json
import re
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


# =========================
# 2. Config
# =========================
# Notebook liegt in THESIS/notebooks/
# Projekt-Root ist daher ein Level höher
NOTEBOOK_DIR = Path.cwd().resolve()
BASE_DIR = NOTEBOOK_DIR.parent

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("BASE_DIR:", BASE_DIR)

INPUT_JSONL = BASE_DIR / "outputs" / "radgraph_input_1500_subset.jsonl"
OUT_DIR = BASE_DIR / "outputs" / "radgraph_extractions_1500_subset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GEN_RAW_JSON_OUT = OUT_DIR / "generated_reports_radgraph_raw.json"
REF_RAW_JSON_OUT = OUT_DIR / "reference_reports_radgraph_raw.json"

GEN_PROCESSED_JSONL_OUT = OUT_DIR / "generated_reports_radgraph_processed.jsonl"
REF_PROCESSED_JSONL_OUT = OUT_DIR / "reference_reports_radgraph_processed.jsonl"

GEN_PROCESSED_CSV_OUT = OUT_DIR / "generated_reports_radgraph_processed.csv"
REF_PROCESSED_CSV_OUT = OUT_DIR / "reference_reports_radgraph_processed.csv"

COMBINED_PROCESSED_JSONL_OUT = OUT_DIR / "combined_radgraph_processed.jsonl"
COMBINED_PROCESSED_CSV_OUT = OUT_DIR / "combined_radgraph_processed.csv"

PAIR_SUMMARY_CSV_OUT = OUT_DIR / "pair_level_summary.csv"


# =========================
# 3. Environment
# =========================
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""


# =========================
# 4. Helper functions
# =========================
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def batched(seq, batch_size):
    for i in range(0, len(seq), batch_size):
        yield seq[i:i + batch_size]


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# =========================
# 5. Compatibility patch for transformers/radgraph
# =========================
import transformers
from transformers import AutoTokenizer, BertTokenizer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

# 1) Fallbacks für ältere Aufrufe
def _encode_plus_compat(self, text=None, text_pair=None, **kwargs):
    return self(text=text, text_pair=text_pair, **kwargs)

def _batch_encode_plus_compat(self, batch_text_or_text_pairs=None, **kwargs):
    return self(batch_text_or_text_pairs, **kwargs)

PreTrainedTokenizerBase.encode_plus = _encode_plus_compat
PreTrainedTokenizerBase.batch_encode_plus = _batch_encode_plus_compat

# 2) BERT-style special token builder
def _bert_build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
    cls = [self.cls_token_id] if self.cls_token_id is not None else []
    sep = [self.sep_token_id] if self.sep_token_id is not None else []

    if token_ids_1 is None:
        return cls + list(token_ids_0) + sep
    return cls + list(token_ids_0) + sep + list(token_ids_1) + sep

def _bert_create_token_type_ids_from_sequences(self, token_ids_0, token_ids_1=None):
    cls = [self.cls_token_id] if self.cls_token_id is not None else []
    sep = [self.sep_token_id] if self.sep_token_id is not None else []

    if token_ids_1 is None:
        return len(cls + list(token_ids_0) + sep) * [0]

    first = cls + list(token_ids_0) + sep
    second = list(token_ids_1) + sep
    return len(first) * [0] + len(second) * [1]

# 3) Patch auf Base + BertTokenizer
PreTrainedTokenizerBase.build_inputs_with_special_tokens = _bert_build_inputs_with_special_tokens
PreTrainedTokenizerBase.create_token_type_ids_from_sequences = _bert_create_token_type_ids_from_sequences

BertTokenizer.build_inputs_with_special_tokens = _bert_build_inputs_with_special_tokens
BertTokenizer.create_token_type_ids_from_sequences = _bert_create_token_type_ids_from_sequences

# 4) Immer slow tokenizer erzwingen
_original_auto_from_pretrained = AutoTokenizer.from_pretrained

def _patched_auto_from_pretrained(*args, **kwargs):
    kwargs["use_fast"] = False
    return _original_auto_from_pretrained(*args, **kwargs)

transformers.AutoTokenizer.from_pretrained = _patched_auto_from_pretrained

# =========================
# 6. Load RadGraph package
# =========================
from radgraph import RadGraph, get_radgraph_processed_annotations


# =========================
# 7. Initialize model
# =========================
radgraph = RadGraph()

# =========================
# 7.5 Load aligned input
# =========================
df = pd.read_json(INPUT_JSONL, lines=True)

print("loaded rows:", len(df))
print("columns:", df.columns.tolist())

# optionale Umbenennung auf deine später verwendeten Namen
df = df.rename(columns={
    "caseid": "case_id",
    "generatedreport": "generated_report",
    "referencereport": "reference_report"
})

required_cols = ["case_id", "generated_report", "reference_report"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")


# =========================
# 8. Run extraction
# =========================
def run_radgraph_on_texts(texts, batch_size=8):
    all_annotations = {}
    global_idx = 0

    for batch in tqdm(list(batched(texts, batch_size)), desc="RadGraph extraction"):
        batch_annotations = radgraph(batch)

        for _, ann in batch_annotations.items():
            all_annotations[str(global_idx)] = ann
            global_idx += 1

    return all_annotations


generated_texts = df["generated_report"].tolist()
reference_texts = df["reference_report"].tolist()

gen_raw = run_radgraph_on_texts(generated_texts, batch_size=8)
ref_raw = run_radgraph_on_texts(reference_texts, batch_size=8)

save_json(gen_raw, GEN_RAW_JSON_OUT)
save_json(ref_raw, REF_RAW_JSON_OUT)

print("saved:", GEN_RAW_JSON_OUT)
print("saved:", REF_RAW_JSON_OUT)


# =========================
# 9. Process annotations
# =========================
def process_single_annotation(case_id, source_type, raw_ann):
    single = {"0": raw_ann}
    processed = get_radgraph_processed_annotations(single)

    processed_rows = []
    processed_annotations = processed.get("processed_annotations", [])

    for item in processed_annotations:
        processed_rows.append({
            "case_id": case_id,
            "source_type": source_type,
            "observation": item.get("observation"),
            "located_at": "; ".join(item.get("located_at", [])) if item.get("located_at") else "",
            "suggestive_of": "; ".join(item.get("suggestive_of", [])) if item.get("suggestive_of") else "",
            "tags": "; ".join(item.get("tags", [])) if item.get("tags") else "",
        })

    return processed_rows, processed


gen_processed_rows = []
ref_processed_rows = []

gen_processed_jsonl_rows = []
ref_processed_jsonl_rows = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Process generated"):
    case_id = row["case_id"]
    raw_ann = gen_raw[str(idx)]

    rows, processed = process_single_annotation(case_id, "generated", raw_ann)
    gen_processed_rows.extend(rows)

    gen_processed_jsonl_rows.append({
        "case_id": case_id,
        "source_type": "generated",
        "raw_annotation": raw_ann,
        "processed_annotation": processed,
    })

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Process reference"):
    case_id = row["case_id"]
    raw_ann = ref_raw[str(idx)]

    rows, processed = process_single_annotation(case_id, "reference", raw_ann)
    ref_processed_rows.extend(rows)

    ref_processed_jsonl_rows.append({
        "case_id": case_id,
        "source_type": "reference",
        "raw_annotation": raw_ann,
        "processed_annotation": processed,
    })

gen_proc_df = pd.DataFrame(gen_processed_rows)
ref_proc_df = pd.DataFrame(ref_processed_rows)

print("generated processed rows:", len(gen_proc_df))
print("reference processed rows:", len(ref_proc_df))

display(gen_proc_df.head(10))
display(ref_proc_df.head(10))


# =========================
# 10. Save processed outputs
# =========================
save_jsonl(gen_processed_jsonl_rows, GEN_PROCESSED_JSONL_OUT)
save_jsonl(ref_processed_jsonl_rows, REF_PROCESSED_JSONL_OUT)

gen_proc_df.to_csv(GEN_PROCESSED_CSV_OUT, index=False)
ref_proc_df.to_csv(REF_PROCESSED_CSV_OUT, index=False)

print("saved:", GEN_PROCESSED_JSONL_OUT)
print("saved:", REF_PROCESSED_JSONL_OUT)
print("saved:", GEN_PROCESSED_CSV_OUT)
print("saved:", REF_PROCESSED_CSV_OUT)


# =========================
# 11. Combined export
# =========================
combined_proc_df = pd.concat([gen_proc_df, ref_proc_df], ignore_index=True)
combined_proc_df.to_csv(COMBINED_PROCESSED_CSV_OUT, index=False)

combined_jsonl_rows = gen_processed_jsonl_rows + ref_processed_jsonl_rows
save_jsonl(combined_jsonl_rows, COMBINED_PROCESSED_JSONL_OUT)

print("saved:", COMBINED_PROCESSED_CSV_OUT)
print("saved:", COMBINED_PROCESSED_JSONL_OUT)

display(combined_proc_df.head(20))


# =========================
# 12. Pair-level summary
# =========================
def unique_obs(df_part, case_id):
    vals = df_part.loc[df_part["case_id"] == case_id, "observation"].dropna().astype(str)
    vals = [v.strip() for v in vals if v.strip() != ""]
    return sorted(set(vals))


summary_rows = []

for case_id in df["case_id"]:
    gen_obs = set(unique_obs(gen_proc_df, case_id))
    ref_obs = set(unique_obs(ref_proc_df, case_id))

    summary_rows.append({
        "case_id": case_id,
        "generated_observation_count": len(gen_obs),
        "reference_observation_count": len(ref_obs),
        "shared_observation_count": len(gen_obs & ref_obs),
        "generated_only_count": len(gen_obs - ref_obs),
        "reference_only_count": len(ref_obs - gen_obs),
        "generated_only": "; ".join(sorted(gen_obs - ref_obs)),
        "reference_only": "; ".join(sorted(ref_obs - gen_obs)),
        "shared_observations": "; ".join(sorted(gen_obs & ref_obs)),
    })

pair_summary_df = pd.DataFrame(summary_rows)
pair_summary_df.to_csv(PAIR_SUMMARY_CSV_OUT, index=False)

print("saved:", PAIR_SUMMARY_CSV_OUT)
display(pair_summary_df.head(10))


# =========================
# 13. Example preview
# =========================
sample_case_ids = df["case_id"].head(5).tolist()

for case_id in sample_case_ids:
    print("=" * 100)
    print("CASE:", case_id)

    print("\nGENERATED REPORT:")
    print(df.loc[df["case_id"] == case_id, "generated_report"].iloc[0])

    print("\nREFERENCE REPORT:")
    print(df.loc[df["case_id"] == case_id, "reference_report"].iloc[0])

    print("\nGENERATED EXTRACTION:")
    display(gen_proc_df[gen_proc_df["case_id"] == case_id])

    print("\nREFERENCE EXTRACTION:")
    display(ref_proc_df[ref_proc_df["case_id"] == case_id])


NOTEBOOK_DIR: /workspace/thesis/notebooks
BASE_DIR: /workspace/thesis
Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


/usr/local/lib/python3.11/dist-packages/transformers/models/bert/tokenization_bert.py:104: DeprecationWarning: Deprecated in 0.9.0: WordPiece.__init__ will not create from files anymore, try `WordPiece.from_file` instead
  self._tokenizer = Tokenizer(WordPiece(self._vocab, unk_token=str(unk_token)))
/usr/local/lib/python3.11/dist-packages/radgraph/radgraph.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.s

loaded rows: 272
columns: ['case_id', 'study_id', 'dicom_id', 'image_path', 'generated_report', 'reference_report', 'hypothesis', 'reference', 'match_source']


RadGraph extraction:   0%|          | 0/34 [00:00<?, ?it/s]

RadGraph extraction:   0%|          | 0/34 [00:00<?, ?it/s]

...

,case_id,source_type,observation,located_at,suggestive_of,tags
11,1026887754137212,reference,congestion,pulmonary vascular,,definitely present
12,1026887754137212,reference,opacities,bibasilar,opacities suggestive of atelectasis; opacities...,definitely present
13,1026887754137212,reference,atelectasis,,,uncertain
14,1026887754137212,reference,infection,,,uncertain


CASE: 1026887757976739

GENERATED REPORT:
In comparison with the study of ___, the monitoring and support devices remain in place. The patient has taken a slightly better inspiration. There is continued enlargement of the cardiac silhouette with evidence of pulmonary vascular congestion. The possibility of supervening pneumonia at the left base would have to be considered in the appropriate clinical setting.

REFERENCE REPORT:
Mild residual retrocardiac opacification remains, pneumonia vs. atelectasis.

GENERATED EXTRACTION:


,case_id,source_type,observation,located_at,suggestive_of,tags
33,1026887757976739,generated,monitoring support devices in place,,,definitely present
34,1026887757976739,generated,patient slightly inspiration,,,definitely present
35,1026887757976739,generated,better,inspiration,,definitely present
36,1026887757976739,generated,enlargement,cardiac silhouette,,definitely present
37,1026887757976739,generated,congestion,pulmonary vascular,,definitely present
38,1026887757976739,generated,supervening pneumonia,left base,,uncertain



REFERENCE EXTRACTION:


,case_id,source_type,observation,located_at,suggestive_of,tags
15,1026887757976739,reference,mild residual opacification,retrocardiac,opacification suggestive of pneumonia; opacifi...,definitely present
16,1026887757976739,reference,pneumonia,,,uncertain
17,1026887757976739,reference,atelectasis,,,uncertain
